# Recurrent Neural Networks (RNNs)

This notebook covers:
- Why RNNs are designed for sequences
- Hidden state and recurrence mechanism
- The vanishing gradient problem
- LSTMs and GRUs as solutions
- Embedding layers for text
- Sentiment analysis on IMDB dataset

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

## Why Sequences Are Different

Text and time series have **sequential structure** where order matters:
- "The movie was not boring" → positive (negation reverses sentiment)
- "The movie was boring" → negative
- A bag-of-words model might confuse these

RNNs maintain a **hidden state** that carries information across timesteps:
```
Input:     [The]    [movie]    [was]    [not]    [boring]
           ↓        ↓          ↓        ↓        ↓
RNN:      h₁ → h₂ → h₃ → h₄ → h₅ (output: sentiment)
```

At each step, the hidden state encodes context from all previous words.

In [ ]:
# Load IMDB sentiment dataset
print("Loading IMDB dataset...")
VOCAB_SIZE = 10000
MAX_LEN = 200

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Example review (first 50 words): {X_train[0][:50]}")
print(f"Labels: {np.unique(y_train)}")

# Pad sequences to same length
from tensorflow.keras.preprocessing import sequence

X_train_padded = sequence.pad_sequences(X_train, maxlen=MAX_LEN)
X_test_padded = sequence.pad_sequences(X_test, maxlen=MAX_LEN)

print(f"After padding: Train {X_train_padded.shape}, Test {X_test_padded.shape}")

## Building an LSTM Model

Architecture:
1. **Embedding**: Convert word indices to dense vectors (learn semantic similarity)
2. **LSTM**: Process sequences, capture long-range dependencies
3. **Dense**: Classification output

LSTM vs vanilla RNN: LSTMs use gating mechanisms to avoid vanishing gradients, allowing learning of long-range dependencies.

In [ ]:
# Build LSTM model
model_lstm = tf.keras.Sequential([
    tf.keras.layers.Embedding(VOCAB_SIZE, 128, input_length=MAX_LEN),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("LSTM Model Architecture:")
model_lstm.summary()

In [ ]:
# Train LSTM
print("Training LSTM on IMDB sentiment...")
history_lstm = model_lstm.fit(
    X_train_padded, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

# Evaluate
test_loss, test_acc = model_lstm.evaluate(X_test_padded, y_test, verbose=0)
print(f"\nLSTM Test Accuracy: {test_acc:.4f}")
print(f"LSTM Test Loss: {test_loss:.4f}")

# Test on sample reviews
word_index = tf.keras.datasets.imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

def decode_review(encoded):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded if i >= 3])

sample_review = decode_review(X_test[0][:100])
print(f"\nSample test review:\n{sample_review}\n")

sample_pred = model_lstm.predict(X_test_padded[:1])
sentiment = "POSITIVE" if sample_pred[0][0] > 0.5 else "NEGATIVE"
confidence = sample_pred[0][0] if sample_pred[0][0] > 0.5 else 1 - sample_pred[0][0]
print(f"Prediction: {sentiment} ({confidence:.2%})")
print(f"Actual: {'POSITIVE' if y_test[0] == 1 else 'NEGATIVE'}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_lstm.history['loss'], label='Training Loss', marker='o', markersize=5)
axes[0].plot(history_lstm.history['val_loss'], label='Validation Loss', marker='s', markersize=5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('LSTM Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_lstm.history['accuracy'], label='Training Accuracy', marker='o', markersize=5)
axes[1].plot(history_lstm.history['val_accuracy'], label='Validation Accuracy', marker='s', markersize=5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('LSTM Training Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("- Embedding layer learns semantic representations of words")
print("- LSTM captures long-range sentence structure (negations, valence shifts)")
print("- Better than bag-of-words because it preserves order")